In [1]:
import torch # pytorch torch
import torchvision # torch vision -> dataset, augmentation, pretrained models...
from torchvision.datasets import MNIST

In [2]:
from torchvision import transforms as T

tfs = T.Compose( [T.Resize((28, 28)), T.Grayscale(num_output_channels = 3), T.ToTensor()] )

ds = MNIST(root = "datasets", train = True, download = True, transform = tfs)
ts_ds = MNIST(root = "datasets", train = False, download = True, transform = tfs)
classes = ds.classes
n_cls = len(classes)
n_cls

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting datasets/MNIST/raw/train-images-idx3-ubyte.gz to datasets/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting datasets/MNIST/raw/train-labels-idx1-ubyte.gz to datasets/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting datasets/MNIST/raw/t10k-images-idx3-ubyte.gz to datasets/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting datasets/MNIST/raw/t10k-labels-idx1-ubyte.gz to datasets/MNIST/raw



10

In [3]:
# PyTorchda DataLoader bilan ishlaymiz
from torch.utils.data import random_split, DataLoader # sklearn.model_selection train_test_split

total_len = len(ds)
tr_len = int(total_len * 0.9) # int
vl_len = total_len - tr_len # int - int = int
tr_len, vl_len

(54000, 6000)

In [4]:
tr_ds, vl_ds = random_split(dataset = ds, lengths=[tr_len, vl_len])
print(len(tr_ds),len(vl_ds))

54000 6000


In [5]:
bs = 128 # mini_batch_size -> data qismlari (mini batch)
tr_dl = DataLoader(dataset = tr_ds, batch_size = bs, shuffle = True, num_workers = 8) # cpu lar soni
vl_dl = DataLoader(dataset = vl_ds, batch_size = bs, shuffle = False, num_workers = 8) # cpu lar soni
ts_dl = DataLoader(dataset = ts_ds, batch_size = 1,  shuffle = False, num_workers = 8) # cpu lar soni
# 0,0,0,0,0,0, ... 1,1,1,1,1,, 2,2,2,2,2,2,2,3,,...9,9,9,9;

/Users/apple/Course/.venv311/lib/python3.11/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


# AL Model


In [6]:
from torch.nn import * # nn -> neural network; * means everything

# Module class o'zgartiradigan functionlari:
# 1) __init__() -> Custom class davomida ishlatiladigan variable larni yaratamiz
# 2) forward
class CustomModel(Module):

  def __init__(self, in_fs, out_fs, n_cls): # initialize
    super().__init__()

    self.lin_layer = Linear(in_features = in_fs, out_features = out_fs)
    self.act = ReLU() # shape o'zgarmaydi
    self.classifier = Linear(in_features = out_fs, out_features = n_cls)


  def forward(self, inp):
    # print("man custom modelga input bo'ldim")
    bs = inp.shape[0]
    inp = inp.view(bs, -1) # 4D -> 2D
    # print(inp.shape)

    out = self.lin_layer(inp)
    out = self.act(out)
    out = self.classifier(out)

    return out

inp = torch.rand(1,3,28,28) # () 28x28 grayscale rasm
# AI modelga har qanday (2D input) input 4D bo'lib kiradi: (bs, im_chs, im_h, im_w)
out_fs = 200
m = CustomModel(in_fs = (inp.shape[1] * inp.shape[2] * inp.shape[3]), out_fs = out_fs, n_cls = n_cls)
xom_bashorat = m(inp)
xom_bashorat
# ehtimolliklar = torch.nn.function(xom_bashorat)

tensor([[ 0.1014,  0.0146,  0.2250,  0.0479, -0.0693,  0.1526, -0.1128, -0.0401,
         -0.0665,  0.0753]], grad_fn=<AddmmBackward0>)

In [7]:
softmax_fn = torch.nn.Softmax(dim = 1)
ehtimolliklar = softmax_fn(xom_bashorat) # bs, logits; bs = 1
ehtimolliklar

tensor([[0.1065, 0.0977, 0.1206, 0.1010, 0.0898, 0.1121, 0.0860, 0.0925, 0.0901,
         0.1038]], grad_fn=<SoftmaxBackward0>)

In [8]:
torch.argmax(ehtimolliklar, dim = 1) # argmax -> eng katta qiymatdagi indeksni qaytaradi

tensor([2])

In [9]:
torch.argmax(xom_bashorat, dim = 1)

tensor([2])

In [10]:
ehtimolliklar.squeeze(0).shape

torch.Size([10])

In [11]:
ehtimolliklar.shape

torch.Size([1, 10])

In [12]:
eht = torch.nn.functional.softmax(xom_bashorat)
eht

/var/folders/cc/66clcqt16ll9b38wnhwpfkkh0000gn/T/ipykernel_2624/2934438553.py:1: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  eht = torch.nn.functional.softmax(xom_bashorat)


tensor([[0.1065, 0.0977, 0.1206, 0.1010, 0.0898, 0.1121, 0.0860, 0.0925, 0.0901,
         0.1038]], grad_fn=<SoftmaxBackward0>)

In [ ]:
# act function:
# 1) linear layer: relu, prelu, leaky relu, ...
# 2) output layer: sigmoid, tanh, softmax

In [ ]:
### Training Loop

In [13]:
# loss function
def mse(pred, target): return (pred - target) ** 2

In [14]:
bashorat = 6
javob = 2

mse(pred = bashorat, target = javob) # 49; 1

16

In [ ]:
# cross entropy loss function; binary: 0, 1 -> 
# 1-holat: javob = 0, bashorat = 1: (1 - 0) * inf = 1 * inf = inf # infinity cheksizlik.
# 2-holat: javob = 1, bashorat = 0: (1 * inf) = inf
# 3-holat: javob = 0, bashorat = 0: 1 * 0 = 0

In [15]:
torch.log(torch.tensor(1))

tensor(0.)

In [16]:
loss_fn = CrossEntropyLoss()
optimizer = torch.optim.Adam(params=m.parameters(), lr = 3e-4) #
# optimization -> trainable parameters value update
# w1 = w0 - lr * w0.grad # Stochastic gradient descent algorithm

In [17]:
eps = 1e-10
1 / (0 + eps) # numerical stability

10000000000.0

In [18]:
javoblar = torch.stack([torch.tensor(4), torch.tensor(5), torch.tensor(2), torch.tensor(1)])
bashoratlar = torch.stack([torch.tensor(4), torch.tensor(0), torch.tensor(1), torch.tensor(1)])
(javoblar == bashoratlar).sum().item()  # nechta data javob bilan tenglini bildiradi.

2

In [19]:
epochs = 10

for epoch in range(epochs):

  if epoch == 1: break
  tr_epoch_acc = 0
  tr_epoch_loss = 0
  vl_epoch_acc = 0
  vl_epoch_loss = 0

  for idx, batch in enumerate(tr_dl):
    if idx == 3: break
    rasmlar, javoblar = batch
    # print(rasmlar.shape) # (bs, im_chs, im_h, im_w)
    # print(javoblar.shape)      # (bs)
    # print(m)
    xom_bashoratlar = m(rasmlar) # logits
    bashoratlar = torch.argmax(xom_bashoratlar, dim = 1)
    # print(xom_bashoratlar.shape) # (bs, n_cls)
    loss = loss_fn(xom_bashoratlar, javoblar) #
    tr_epoch_loss += loss.item()

    optimizer.zero_grad() # gradientlarni reset qiladi
    loss.backward() # backpropagationdan -> oldin doimo optimizer.zero_grad() chiqarish kerak bulma qiymatlar biri biriga tasir kursata boshlaydi.
    optimizer.step() # optimization

    batch_acc = (bashoratlar == javoblar).sum().item() # nechta data javob bilan tenglini bildiradi
    tr_epoch_acc += batch_acc # nechta data javob bilan tengligini bildiradi.

  print(f"{epoch + 1}-epochdagi train loss -> {tr_epoch_loss / len(tr_dl):.3f}") # batch soniga bo'linadi
  print(f"{epoch + 1}-epochdagi train acc  -> {tr_epoch_acc / len(tr_dl.dataset):.3f}") # element wise bo'lganiga datalar soniga bo'linadi


  with torch.no_grad():  # validation vaqtida gradoentlar hisoblanmaydi, optimizer ishlatilmaydi, modelni train holatiga o'tkazish shart emas.
    for idx, batch in enumerate(vl_dl):
      if idx == 3: break
      rasmlar, javoblar = batch
      xom_bashoratlar = m(rasmlar) # logits
      bashoratlar = torch.argmax(xom_bashoratlar, dim = 1)
      loss = loss_fn(xom_bashoratlar, javoblar)
      vl_epoch_loss += loss.item()

      batch_acc = (bashoratlar == javoblar).sum().item() # nechta data javob bilan tenglini bildiradi
      vl_epoch_acc += batch_acc

  print(f"{epoch + 1}-epochdagi validation loss -> {vl_epoch_loss / len(vl_dl):.3f}") # batch soniga bo'linadi
  print(f"{epoch + 1}-epochdagi validation acc -> {vl_epoch_acc / len(vl_dl.dataset):.3f}") # element wise bo'lganiga datalar soniga bo'linadi


1-epochdagi train loss -> 0.016
1-epochdagi train acc  -> 0.002
1-epochdagi validation loss -> 0.137
1-epochdagi validation acc -> 0.028
